# Equimpent Failure Prediction


You must do this step by step:

1️⃣ Data Understanding

• What does each column represent?
• Which columns look problematic or messy?
• Which ones are likely predictive?


# 1. Pressure should not be date value format

In [288]:
import pandas as pd 
import numpy as np 
import sklearn as sk 
import math 
import string as st 


In [289]:
path = r"C:\Users\ritaj\Downloads\Vaillant_sample_data.csv"
df = pd.read_csv(path,delimiter=';')

In [290]:
df

,customer_id,device_id,install_date,last_service,city,avg_temp,pressure,noise_level,error_code,service_calls_12m,contract_type,failure_next_30d
0,C001,DV1001,10.02.2019,15.01.2024,berlin,72,1.08.2026,low,E101,2,Premium,0
1,C002,DV1002,10.03.2020,NaN,Hamburg,85,2.03.2026,high,NaN,5,Basic,1
2,C003,DV1003,1.07.2018,20.11.2023,Munich,#NUM!,1.05.2026,medium,E203,1,Premium,0
3,C004,DV1004,30.05.2017,10.08.2022,berlin,95,2.09.2026,high,E101,7,Basic,1
4,C005,DV1005,1.12.2021,1.01.2024,Stuttgart,60,NaN,low,NaN,0,Premium,0
5,C006,DV1006,1.01.2016,5.10.2023,Frankfurt,105,3.02.2026,high,E404,8,Basic,1
6,C007,DV1007,10.09.2019,1.02.2024,Hamburg,70,1.07.2026,low,E101,2,Premium,0
7,C008,DV1008,20.04.2015,15.05.2022,Munich,110,3.05.2026,high,E404,9,Basic,1
8,C009,DV1009,30.06.2020,1.12.2023,berlin,75,1.09.2026,medium,NaN,3,Premium,0
9,C010,DV1010,11.11.2018,NaN,Stuttgart,90,2.06.2026,high,E203,6,Basic,1


In [291]:
# Let us see the general information of the data
# 1. see the datatype we have per each columns
# 2. the number of null values per each columns
# 3. the general distribution of data
# 4. check for the outlier in the data from max and min values

In [292]:
df.columns

Index(['customer_id', 'device_id', 'install_date', 'last_service', 'city',
       'avg_temp', 'pressure', 'noise_level', 'error_code',
       'service_calls_12m', 'contract_type', 'failure_next_30d'],
      dtype='object')

In [293]:
df['customer_id'].str.len().unique()

array([4])

In [294]:
df['device_id'].str.len().unique()

array([6])

In [295]:
df['error_code'].str.len().unique()

array([ 4., nan])

In [296]:
df.dtypes

customer_id          object
device_id            object
install_date         object
last_service         object
city                 object
avg_temp             object
pressure             object
noise_level          object
error_code           object
service_calls_12m     int64
contract_type        object
failure_next_30d      int64
dtype: object

In [297]:
df.isna().sum()

customer_id          0
device_id            0
install_date         0
last_service         2
city                 0
avg_temp             0
pressure             1
noise_level          0
error_code           3
service_calls_12m    0
contract_type        0
failure_next_30d     0
dtype: int64

In [298]:
for i in df.columns:
    print({i:df[i].unique()})

{'customer_id': array(['C001', 'C002', 'C003', 'C004', 'C005', 'C006', 'C007', 'C008',
       'C009', 'C010'], dtype=object)}
{'device_id': array(['DV1001', 'DV1002', 'DV1003', 'DV1004', 'DV1005', 'DV1006',
       'DV1007', 'DV1008', 'DV1009', 'DV1010'], dtype=object)}
{'install_date': array(['10.02.2019', '10.03.2020', '1.07.2018', '30.05.2017', '1.12.2021',
       '1.01.2016', '10.09.2019', '20.04.2015', '30.06.2020',
       '11.11.2018'], dtype=object)}
{'last_service': array(['15.01.2024', nan, '20.11.2023', '10.08.2022', '1.01.2024',
       '5.10.2023', '1.02.2024', '15.05.2022', '1.12.2023'], dtype=object)}
{'city': array(['berlin', 'Hamburg', 'Munich', 'Stuttgart', 'Frankfurt'],
      dtype=object)}
{'avg_temp': array(['72', '85', '#NUM!', '95', '60', '105', '70', '110', '75', '90'],
      dtype=object)}
{'pressure': array(['1.08.2026', '2.03.2026', '1.05.2026', '2.09.2026', nan,
       '3.02.2026', '1.07.2026', '3.05.2026', '1.09.2026', '2.06.2026'],
      dtype=object)}
{'nois

1. follow up question:- what is service call? is it the frequency of the client call in last 12 month? just want to verify

# Data Cleaning Approach

# 1. Let us handle the missing values

In [299]:
# let us try to change the whole data types of columns with automatic handling

In [300]:
df = df.convert_dtypes(infer_objects=True)

In [301]:
df.dtypes

customer_id          string[python]
device_id            string[python]
install_date         string[python]
last_service         string[python]
city                 string[python]
avg_temp             string[python]
pressure             string[python]
noise_level          string[python]
error_code           string[python]
service_calls_12m             Int64
contract_type        string[python]
failure_next_30d              Int64
dtype: object

In [302]:
# let us take pressure
df['pressure'].unique()

<StringArray>
['1.08.2026', '2.03.2026', '1.05.2026', '2.09.2026',        <NA>, '3.02.2026',
 '1.07.2026', '3.05.2026', '1.09.2026', '2.06.2026']
Length: 10, dtype: string

In [303]:
df.columns[df.isna().sum()>=1]

Index(['last_service', 'pressure', 'error_code'], dtype='object')

In [304]:
df.loc[df['last_service'].isna()==True]

,customer_id,device_id,install_date,last_service,city,avg_temp,pressure,noise_level,error_code,service_calls_12m,contract_type,failure_next_30d
1,C002,DV1002,10.03.2020,<NA>,Hamburg,85,2.03.2026,high,<NA>,5,Basic,1
9,C010,DV1010,11.11.2018,<NA>,Stuttgart,90,2.06.2026,high,E203,6,Basic,1


In [305]:
df.loc[df['error_code'].isna()==True]

,customer_id,device_id,install_date,last_service,city,avg_temp,pressure,noise_level,error_code,service_calls_12m,contract_type,failure_next_30d
1,C002,DV1002,10.03.2020,<NA>,Hamburg,85,2.03.2026,high,<NA>,5,Basic,1
4,C005,DV1005,1.12.2021,1.01.2024,Stuttgart,60,<NA>,low,<NA>,0,Premium,0
8,C009,DV1009,30.06.2020,1.12.2023,berlin,75,1.09.2026,medium,<NA>,3,Premium,0


In [306]:
df['error_code']=df['error_code'].fillna(0)

In [307]:
df['error_code'].isna().sum()

np.int64(0)

In [308]:
df['pressure']

0    1.08.2026
1    2.03.2026
2    1.05.2026
3    2.09.2026
4         <NA>
5    3.02.2026
6    1.07.2026
7    3.05.2026
8    1.09.2026
9    2.06.2026
Name: pressure, dtype: string

In [309]:
q=df['pressure'].str.split('.',expand=True)[:]

In [310]:
q=q[[0,1]]

In [311]:
q.fillna(np.nan,inplace=True)

In [312]:
q

,0,1
0,1,08
1,2,03
2,1,05
3,2,09
4,<NA>,<NA>
5,3,02
6,1,07
7,3,05
8,1,09
9,2,06


In [313]:
q[0]=pd.to_numeric(q[0].values)
q[1]=pd.to_numeric(q[1].values)
q[1]=q[1]/10

In [314]:
q[0]=pd.to_numeric(q[0].values)

In [315]:
q.dtypes

0      Int64
1    Float64
dtype: object

In [316]:
df['pressure'] = q[0] + q[1]

In [317]:
df['pressure']=df['pressure'].astype('float64')

In [318]:
from sklearn.impute import SimpleImputer 
imput = SimpleImputer(strategy='median')
X=df[['pressure']]

In [319]:
imput.fit(X)

,missing_values,nan
,strategy,'median'
,fill_value,None
,copy,True
,add_indicator,False
,keep_empty_features,False


In [320]:
X_new = imput.transform(X)

In [321]:
df['pressure'] = X_new

In [322]:
df.drop(columns='error_code',inplace=True)

In [323]:
df

,customer_id,device_id,install_date,last_service,city,avg_temp,pressure,noise_level,service_calls_12m,contract_type,failure_next_30d
0,C001,DV1001,10.02.2019,15.01.2024,berlin,72,1.8,low,2,Premium,0
1,C002,DV1002,10.03.2020,<NA>,Hamburg,85,2.3,high,5,Basic,1
2,C003,DV1003,1.07.2018,20.11.2023,Munich,#NUM!,1.5,medium,1,Premium,0
3,C004,DV1004,30.05.2017,10.08.2022,berlin,95,2.9,high,7,Basic,1
4,C005,DV1005,1.12.2021,1.01.2024,Stuttgart,60,2.3,low,0,Premium,0
5,C006,DV1006,1.01.2016,5.10.2023,Frankfurt,105,3.2,high,8,Basic,1
6,C007,DV1007,10.09.2019,1.02.2024,Hamburg,70,1.7,low,2,Premium,0
7,C008,DV1008,20.04.2015,15.05.2022,Munich,110,3.5,high,9,Basic,1
8,C009,DV1009,30.06.2020,1.12.2023,berlin,75,1.9,medium,3,Premium,0
9,C010,DV1010,11.11.2018,<NA>,Stuttgart,90,2.6,high,6,Basic,1


In [324]:
df['install_date'] = pd.to_datetime(df['install_date'],format='%d.%m.%Y')
df['last_service'] = pd.to_datetime(df['last_service'],format='%d.%m.%Y')


In [325]:
from datetime import datetime as dt
from datetime import timedelta


In [326]:
# take some kind of naive solution where the average level of temperature

In [327]:
df

,customer_id,device_id,install_date,last_service,city,avg_temp,pressure,noise_level,service_calls_12m,contract_type,failure_next_30d
0,C001,DV1001,2019-02-10,2024-01-15,berlin,72,1.8,low,2,Premium,0
1,C002,DV1002,2020-03-10,NaT,Hamburg,85,2.3,high,5,Basic,1
2,C003,DV1003,2018-07-01,2023-11-20,Munich,#NUM!,1.5,medium,1,Premium,0
3,C004,DV1004,2017-05-30,2022-08-10,berlin,95,2.9,high,7,Basic,1
4,C005,DV1005,2021-12-01,2024-01-01,Stuttgart,60,2.3,low,0,Premium,0
5,C006,DV1006,2016-01-01,2023-10-05,Frankfurt,105,3.2,high,8,Basic,1
6,C007,DV1007,2019-09-10,2024-02-01,Hamburg,70,1.7,low,2,Premium,0
7,C008,DV1008,2015-04-20,2022-05-15,Munich,110,3.5,high,9,Basic,1
8,C009,DV1009,2020-06-30,2023-12-01,berlin,75,1.9,medium,3,Premium,0
9,C010,DV1010,2018-11-11,NaT,Stuttgart,90,2.6,high,6,Basic,1


In [328]:
df['avg_temp'].replace({'#NUM!':'0'},inplace=True)

C:\Users\ritaj\AppData\Local\Temp\ipykernel_24360\991953160.py:1: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df['avg_temp'].replace({'#NUM!':'0'},inplace=True)


In [329]:
df

,customer_id,device_id,install_date,last_service,city,avg_temp,pressure,noise_level,service_calls_12m,contract_type,failure_next_30d
0,C001,DV1001,2019-02-10,2024-01-15,berlin,72,1.8,low,2,Premium,0
1,C002,DV1002,2020-03-10,NaT,Hamburg,85,2.3,high,5,Basic,1
2,C003,DV1003,2018-07-01,2023-11-20,Munich,0,1.5,medium,1,Premium,0
3,C004,DV1004,2017-05-30,2022-08-10,berlin,95,2.9,high,7,Basic,1
4,C005,DV1005,2021-12-01,2024-01-01,Stuttgart,60,2.3,low,0,Premium,0
5,C006,DV1006,2016-01-01,2023-10-05,Frankfurt,105,3.2,high,8,Basic,1
6,C007,DV1007,2019-09-10,2024-02-01,Hamburg,70,1.7,low,2,Premium,0
7,C008,DV1008,2015-04-20,2022-05-15,Munich,110,3.5,high,9,Basic,1
8,C009,DV1009,2020-06-30,2023-12-01,berlin,75,1.9,medium,3,Premium,0
9,C010,DV1010,2018-11-11,NaT,Stuttgart,90,2.6,high,6,Basic,1


In [330]:
df['avg_temp'] = pd.to_numeric(df['avg_temp'])

In [331]:
df[df['noise_level']=='medium']['avg_temp']

2     0
8    75
Name: avg_temp, dtype: Int64

In [332]:
df['avg_temp'].replace({0:df['avg_temp'].mean().round(decimals=0)},inplace=True)

C:\Users\ritaj\AppData\Local\Temp\ipykernel_24360\394684549.py:1: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df['avg_temp'].replace({0:df['avg_temp'].mean().round(decimals=0)},inplace=True)


In [333]:
df

,customer_id,device_id,install_date,last_service,city,avg_temp,pressure,noise_level,service_calls_12m,contract_type,failure_next_30d
0,C001,DV1001,2019-02-10,2024-01-15,berlin,72,1.8,low,2,Premium,0
1,C002,DV1002,2020-03-10,NaT,Hamburg,85,2.3,high,5,Basic,1
2,C003,DV1003,2018-07-01,2023-11-20,Munich,76,1.5,medium,1,Premium,0
3,C004,DV1004,2017-05-30,2022-08-10,berlin,95,2.9,high,7,Basic,1
4,C005,DV1005,2021-12-01,2024-01-01,Stuttgart,60,2.3,low,0,Premium,0
5,C006,DV1006,2016-01-01,2023-10-05,Frankfurt,105,3.2,high,8,Basic,1
6,C007,DV1007,2019-09-10,2024-02-01,Hamburg,70,1.7,low,2,Premium,0
7,C008,DV1008,2015-04-20,2022-05-15,Munich,110,3.5,high,9,Basic,1
8,C009,DV1009,2020-06-30,2023-12-01,berlin,75,1.9,medium,3,Premium,0
9,C010,DV1010,2018-11-11,NaT,Stuttgart,90,2.6,high,6,Basic,1


In [334]:
# let us take naive solution where if last 12 month if they call 5 times, let us take as uniformity in the duration of failure

final_time = pd.to_datetime('2024-12-31',format='%Y-%m-%d')
failure_5 = 12/5*30
failure_6=12/6*30



In [335]:
df.loc[df['device_id']=='DV1010','last_service'] = final_time-timedelta(days=failure_5)
df.loc[df['device_id']=='DV1002','last_service'] = final_time-timedelta(days=failure_5)

In [336]:
df

,customer_id,device_id,install_date,last_service,city,avg_temp,pressure,noise_level,service_calls_12m,contract_type,failure_next_30d
0,C001,DV1001,2019-02-10,2024-01-15,berlin,72,1.8,low,2,Premium,0
1,C002,DV1002,2020-03-10,2024-10-20,Hamburg,85,2.3,high,5,Basic,1
2,C003,DV1003,2018-07-01,2023-11-20,Munich,76,1.5,medium,1,Premium,0
3,C004,DV1004,2017-05-30,2022-08-10,berlin,95,2.9,high,7,Basic,1
4,C005,DV1005,2021-12-01,2024-01-01,Stuttgart,60,2.3,low,0,Premium,0
5,C006,DV1006,2016-01-01,2023-10-05,Frankfurt,105,3.2,high,8,Basic,1
6,C007,DV1007,2019-09-10,2024-02-01,Hamburg,70,1.7,low,2,Premium,0
7,C008,DV1008,2015-04-20,2022-05-15,Munich,110,3.5,high,9,Basic,1
8,C009,DV1009,2020-06-30,2023-12-01,berlin,75,1.9,medium,3,Premium,0
9,C010,DV1010,2018-11-11,2024-10-20,Stuttgart,90,2.6,high,6,Basic,1


# Feature Engineering

This section will be used to create a derived columns from the original data that we have !

In [337]:
import datetime as dt 
final_time = pd.to_datetime('2024-12-31',format='%Y-%m-%d')
df['age_of_device']=final_time-df['install_date']

In [338]:
q=df['age_of_device'].astype(str).str.split(expand=True)

In [339]:
q[0]= (pd.to_numeric(q[0])/365)

In [340]:
df['age_of_device']=q[0]

In [341]:
df

,customer_id,device_id,install_date,last_service,city,avg_temp,pressure,noise_level,service_calls_12m,contract_type,failure_next_30d,age_of_device
0,C001,DV1001,2019-02-10,2024-01-15,berlin,72,1.8,low,2,Premium,0,5.893151
1,C002,DV1002,2020-03-10,2024-10-20,Hamburg,85,2.3,high,5,Basic,1,4.813699
2,C003,DV1003,2018-07-01,2023-11-20,Munich,76,1.5,medium,1,Premium,0,6.506849
3,C004,DV1004,2017-05-30,2022-08-10,berlin,95,2.9,high,7,Basic,1,7.594521
4,C005,DV1005,2021-12-01,2024-01-01,Stuttgart,60,2.3,low,0,Premium,0,3.084932
5,C006,DV1006,2016-01-01,2023-10-05,Frankfurt,105,3.2,high,8,Basic,1,9.005479
6,C007,DV1007,2019-09-10,2024-02-01,Hamburg,70,1.7,low,2,Premium,0,5.312329
7,C008,DV1008,2015-04-20,2022-05-15,Munich,110,3.5,high,9,Basic,1,9.706849
8,C009,DV1009,2020-06-30,2023-12-01,berlin,75,1.9,medium,3,Premium,0,4.506849
9,C010,DV1010,2018-11-11,2024-10-20,Stuttgart,90,2.6,high,6,Basic,1,6.142466


In [342]:
 df = pd.get_dummies(df,columns=['noise_level','contract_type'])

In [343]:
df.columns

Index(['customer_id', 'device_id', 'install_date', 'last_service', 'city',
       'avg_temp', 'pressure', 'service_calls_12m', 'failure_next_30d',
       'age_of_device', 'noise_level_high', 'noise_level_low',
       'noise_level_medium', 'contract_type_Basic', 'contract_type_Premium'],
      dtype='object')

In [344]:
bool_col = ['noise_level_high','noise_level_low', 'noise_level_medium',
       'contract_type_Premium','contract_type_Basic']


df[bool_col]  = df[bool_col].astype(str)


df[bool_col]  = df[bool_col].replace({'False': 0, 'True': 1,'<NA>':'unknown'})


C:\Users\ritaj\AppData\Local\Temp\ipykernel_24360\1994195783.py:8: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df[bool_col]  = df[bool_col].replace({'False': 0, 'True': 1,'<NA>':'unknown'})


In [345]:
df.columns

Index(['customer_id', 'device_id', 'install_date', 'last_service', 'city',
       'avg_temp', 'pressure', 'service_calls_12m', 'failure_next_30d',
       'age_of_device', 'noise_level_high', 'noise_level_low',
       'noise_level_medium', 'contract_type_Basic', 'contract_type_Premium'],
      dtype='object')

In [346]:
from sklearn.preprocessing import StandardScaler

In [347]:
scaler= StandardScaler()
x=['avg_temp', 'pressure', 'service_calls_12m']
scaled_temp=scaler.fit_transform(df[['avg_temp']])
scaled_pres=scaler.fit_transform(df[['pressure']])
scaled_call=scaler.fit_transform(df[['service_calls_12m']])

In [348]:
df['avg_temp']= scaled_temp
df['pressure'] = scaled_pres
df['service_calls_12m'] = scaled_call

In [349]:
df

,customer_id,device_id,install_date,last_service,city,avg_temp,pressure,service_calls_12m,failure_next_30d,age_of_device,noise_level_high,noise_level_low,noise_level_medium,contract_type_Basic,contract_type_Premium
0,C001,DV1001,2019-02-10,2024-01-15,berlin,-0.772116,-0.894455,-0.774890,0,5.893151,0,1,0,0,1
1,C002,DV1002,2020-03-10,2024-10-20,Hamburg,0.078520,-0.109845,0.235836,1,4.813699,1,0,0,1,0
2,C003,DV1003,2018-07-01,2023-11-20,Munich,-0.510382,-1.365220,-1.111798,0,6.506849,0,0,1,0,1
3,C004,DV1004,2017-05-30,2022-08-10,berlin,0.732856,0.831686,0.909653,1,7.594521,1,0,0,1,0
4,C005,DV1005,2021-12-01,2024-01-01,Stuttgart,-1.557320,-0.109845,-1.448707,0,3.084932,0,1,0,0,1
5,C006,DV1006,2016-01-01,2023-10-05,Frankfurt,1.387192,1.302452,1.246562,1,9.005479,1,0,0,1,0
6,C007,DV1007,2019-09-10,2024-02-01,Hamburg,-0.902984,-1.051377,-0.774890,0,5.312329,0,1,0,0,1
7,C008,DV1008,2015-04-20,2022-05-15,Munich,1.714360,1.773217,1.583470,1,9.706849,1,0,0,1,0
8,C009,DV1009,2020-06-30,2023-12-01,berlin,-0.575816,-0.737533,-0.437981,0,4.506849,0,0,1,0,1
9,C010,DV1010,2018-11-11,2024-10-20,Stuttgart,0.405688,0.360920,0.572745,1,6.142466,1,0,0,1,0


In [350]:
X=df[['avg_temp', 'pressure',
       'age_of_device', 'noise_level_high', 'noise_level_low',
       'noise_level_medium', 'contract_type_Basic', 'contract_type_Premium']]
y=df['failure_next_30d']

In [351]:
X

,avg_temp,pressure,age_of_device,noise_level_high,noise_level_low,noise_level_medium,contract_type_Basic,contract_type_Premium
0,-0.772116,-0.894455,5.893151,0,1,0,0,1
1,0.078520,-0.109845,4.813699,1,0,0,1,0
2,-0.510382,-1.365220,6.506849,0,0,1,0,1
3,0.732856,0.831686,7.594521,1,0,0,1,0
4,-1.557320,-0.109845,3.084932,0,1,0,0,1
5,1.387192,1.302452,9.005479,1,0,0,1,0
6,-0.902984,-1.051377,5.312329,0,1,0,0,1
7,1.714360,1.773217,9.706849,1,0,0,1,0
8,-0.575816,-0.737533,4.506849,0,0,1,0,1
9,0.405688,0.360920,6.142466,1,0,0,1,0


# Predict the failure the next month failure

In [352]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report

In [353]:
X_train,X_test,y_train,y_test=train_test_split(X,y,test_size=0.1)

In [354]:
reg = LogisticRegression()

In [355]:
reg.fit(X_train,y_train)

,penalty,'l2'
,dual,False
,tol,0.0001
,C,1.0
,fit_intercept,True
,intercept_scaling,1
,class_weight,None
,random_state,None
,solver,'lbfgs'
,max_iter,100
,multi_class,'deprecated'


In [356]:
reg.predict(X_test)

array([0.])

In [357]:
y_test

0    0
Name: failure_next_30d, dtype: Int64

In [358]:
pred = reg.predict(X)

In [359]:
df=pd.DataFrame({'Prediction':pred,'Original':y})

In [365]:
df

,Prediction,Original
0,0.0,0
1,1.0,1
2,0.0,0
3,1.0,1
4,0.0,0
5,1.0,1
6,0.0,0
7,1.0,1
8,0.0,0
9,1.0,1


In [366]:
print(classification_report(df['Original'],df['Prediction']))

              precision    recall  f1-score   support

         0.0       1.00      1.00      1.00         5
         1.0       1.00      1.00      1.00         5

    accuracy                           1.00        10
   macro avg       1.00      1.00      1.00        10
weighted avg       1.00      1.00      1.00        10



In [362]:
X

,avg_temp,pressure,age_of_device,noise_level_high,noise_level_low,noise_level_medium,contract_type_Basic,contract_type_Premium
0,-0.772116,-0.894455,5.893151,0,1,0,0,1
1,0.078520,-0.109845,4.813699,1,0,0,1,0
2,-0.510382,-1.365220,6.506849,0,0,1,0,1
3,0.732856,0.831686,7.594521,1,0,0,1,0
4,-1.557320,-0.109845,3.084932,0,1,0,0,1
5,1.387192,1.302452,9.005479,1,0,0,1,0
6,-0.902984,-1.051377,5.312329,0,1,0,0,1
7,1.714360,1.773217,9.706849,1,0,0,1,0
8,-0.575816,-0.737533,4.506849,0,0,1,0,1
9,0.405688,0.360920,6.142466,1,0,0,1,0


In [367]:
X.columns

Index(['avg_temp', 'pressure', 'age_of_device', 'noise_level_high',
       'noise_level_low', 'noise_level_medium', 'contract_type_Basic',
       'contract_type_Premium'],
      dtype='object')

In [368]:
np.exp(reg.coef_)

array([[1.79905194, 1.82632935, 1.37670371, 1.6903437 , 0.80672388,
        0.73298489, 1.6903437 , 0.59131642]])

In [369]:
df

,Prediction,Original
0,0.0,0
1,1.0,1
2,0.0,0
3,1.0,1
4,0.0,0
5,1.0,1
6,0.0,0
7,1.0,1
8,0.0,0
9,1.0,1
